In [66]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

In [67]:
spark = SparkSession.builder \
    .appName("SmartRetailAnalytics") \
    .config("spark.sql.warehouse.dir", "spark-warehouse") \
    .getOrCreate()

In [68]:
os.makedirs("raw_data", exist_ok=True)

In [69]:
with open("raw_data/stores.csv", "w") as f:
    f.write("""store_id,store_name,city,state,store_type,manager_name
S101,Metro Mart Hyderabad,Hyderabad,Telangana,Supermarket,Rahul Sharma
S102,Metro Mart Bangalore,Bangalore,Karnataka,Supermarket,Priya Reddy
S103,Metro Mart Mumbai,Mumbai,Maharashtra,Hypermarket,Amit Kumar
S104,Metro Mart Chennai,Chennai,Tamil Nadu,Supermarket,Sneha Patel
S105,Metro Mart Delhi,Delhi,Delhi,Hypermarket,Farhan Ali
S106,Metro Mart Pune,Pune,Maharashtra,Mini Store,Neha Singh
S107,Metro Mart Kochi,Kochi,Kerala,Mini Store,Arjun Verma
S108,Metro Mart Jaipur,Jaipur,Rajasthan,Supermarket,Meera Nair""")

In [70]:
with open("raw_data/products.csv", "w") as f:
    f.write("""product_id,product_name,category,brand,supplier_id,unit_price
P101,Laptop,Electronics,Lenovo,S201,65000
P102,Mobile,Electronics,Samsung,S202,25000
P103,Television,Electronics,LG,S203,45000
P104,Office Chair,Furniture,Featherlite,S204,7000
P105,Study Table,Furniture,Urban Ladder,S204,12000
P106,Shoes,Fashion,Nike,S205,4500
P107,Watch,Fashion,Fastrack,S206,8000
P108,Backpack,Fashion,Wildcraft,S206,2500
P109,Refrigerator,Electronics,Whirlpool,S203,38000
P110,Sofa,Furniture,Godrej,S204,32000
P111,Headphones,Electronics,Sony,S999,3000
P112,T-Shirt,Fashion,Puma,,1500""")

In [71]:
with open("raw_data/inventory.csv", "w") as f:
    f.write("""inventory_id,store_id,product_id,stock_quantity,reorder_level,last_update
I1001,S101,P101,10,5,2026-01-10
I1002,S101,P102,25,10,2026-01-10
I1003,S101,P104,3,5,2026-01-11
I1004,S102,P101,8,5,2026-01-12
I1005,S102,P103,5,4,2026-01-12
I1006,S103,P105,2,5,2026-01-13
I1007,S103,P106,30,10,2026-01-14
I1008,S104,P107,4,5,2026-01-15
I1009,S105,P108,50,20,2026-01-15
I1010,S106,P109,,6,2026-01-16
I1011,S107,P110,1,3,2026-01-17
I1012,S108,P120,12,5,2026-01-18""")

In [72]:
with open("raw_data/sales.csv", "w") as f:
    f.write("""sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_mode
SA1001,S101,P101,2026-01-10,1,65000,UPI
SA1002,S101,P102,2026-01-10,2,50000,Card
SA1003,S102,P101,2026-01-11,1,65000,UPI
SA1004,S103,P106,2026-01-12,4,18000,Cash
SA1005,S104,P107,2026-01-12,1,8000,Card
SA1006,S105,P108,2026-01-13,5,12500,UPI
SA1007,S106,P109,2026-01-14,1,38000,Card
SA1008,S107,P110,2026-01-15,1,32000,UPI
SA1009,S108,P120,2026-01-15,2,10000,Cash
SA1010,S101,P104,2026-01-16,2,14000,
SA1011,S102,P103,2026-01-17,1,,UPI
SA1012,S103,P105,2026-01-18,1,12000,Card
SA1013,S104,P107,2026-02-01,2,16000,UPI
SA1014,S105,P108,2026-02-02,3,7500,Cash
SA1015,S101,P102,2026-02-03,1,25000,Card""")

In [73]:
with open("raw_data/suppliers.json", "w") as f:
    f.write("""[
  {
    "supplier_id": "S201",
    "supplier_name": "TechSource India",
    "city": "Hyderabad",
    "rating": 4.5,
    "contact": { "phone": "9876500011", "email": "techsource@mail.com" }
  },
  {
    "supplier_id": "S202",
    "supplier_name": "MobileWorld Distributors",
    "city": "Bangalore",
    "rating": 4.2,
    "contact": { "phone": null, "email": "mobileworld@mail.com" }
  },
  {
    "supplier_id": "S203",
    "supplier_name": "HomeTech Supply",
    "city": "Mumbai",
    "rating": 4.4,
    "contact": { "phone": "9876500013", "email": null }
  },
  {
    "supplier_id": "S204",
    "supplier_name": "Urban Furniture Co",
    "city": "Delhi",
    "rating": 4.0,
    "contact": { "phone": "9876500014", "email": "urban@mail.com" }
  },
  {
    "supplier_id": "S205",
    "supplier_name": "Fashion Direct",
    "city": "Pune",
    "rating": 3.8,
    "contact": { "phone": null, "email": null }
  }
]""")

In [74]:
print("=== Part 1: Ingestion ===")

=== Part 1: Ingestion ===


In [75]:
df_stores_raw = spark.read.csv("raw_data/stores.csv", header=True, inferSchema=True)
df_products_raw = spark.read.csv("raw_data/products.csv", header=True, inferSchema=True)
df_inventory_raw = spark.read.csv("raw_data/inventory.csv", header=True, inferSchema=True)
df_sales_raw = spark.read.csv("raw_data/sales.csv", header=True, inferSchema=True)

In [76]:
df_suppliers_raw = spark.read.option("multiLine", "true").json("raw_data/suppliers.json")

In [77]:
print("\n--- 6. Schemas ---")
df_stores_raw.printSchema()
df_products_raw.printSchema()
df_inventory_raw.printSchema()
df_sales_raw.printSchema()
df_suppliers_raw.printSchema()


--- 6. Schemas ---
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 

In [78]:
print("\n--- 7. Record Counts ---")
print(f"Stores Raw: {df_stores_raw.count()}")
print(f"Products Raw: {df_products_raw.count()}")
print(f"Inventory Raw: {df_inventory_raw.count()}")
print(f"Sales Raw: {df_sales_raw.count()}")
print(f"Suppliers Raw: {df_suppliers_raw.count()}")


--- 7. Record Counts ---
Stores Raw: 8
Products Raw: 12
Inventory Raw: 12
Sales Raw: 15
Suppliers Raw: 5


In [79]:
df_stores_raw.write.mode("overwrite").parquet("bronze/stores")
df_products_raw.write.mode("overwrite").parquet("bronze/products")
df_inventory_raw.write.mode("overwrite").parquet("bronze/inventory")
df_sales_raw.write.mode("overwrite").parquet("bronze/sales")
df_suppliers_raw.write.mode("overwrite").parquet("bronze/suppliers")

In [80]:
print("\n=== Part 2: Data Cleaning ===")


=== Part 2: Data Cleaning ===


In [81]:
# Reload from Bronze to decouple structural transformations
df_stores_b = spark.read.parquet("bronze/stores")
df_products_b = spark.read.parquet("bronze/products")
df_inventory_b = spark.read.parquet("bronze/inventory")
df_sales_b = spark.read.parquet("bronze/sales")

In [82]:
print("\n--- 9. Missing Supplier IDs ---")
df_products_b.filter(F.col("supplier_id").isNull() | (F.col("supplier_id") == "")).show()


--- 9. Missing Supplier IDs ---
+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [83]:

df_inventory_b.filter(F.col("stock_quantity").isNull()).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [84]:
df_sales_b.filter(F.col("sale_amount").isNull()).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|         UPI|
+-------+--------+----------+----------+-------------+-----------+------------+



In [85]:
df_sales_b.filter(F.col("payment_mode").isNull() | (F.trim(F.col("payment_mode")) == "")).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|        NULL|
+-------+--------+----------+----------+-------------+-----------+------------+



In [86]:
df_products_silver = df_products_b.withColumn(
    "supplier_id", F.when(F.col("supplier_id").isNull(), "UNKNOWN").otherwise(F.col("supplier_id"))
).withColumn(
    "data_quality_status", F.when(F.col("supplier_id") == "UNKNOWN", "FLAGGED").otherwise("VALID")
)

df_inventory_silver = df_inventory_b.withColumn(
    "stock_quantity", F.when(F.col("stock_quantity").isNull(), 0).otherwise(F.col("stock_quantity"))
).withColumn(
    "data_quality_status", F.when(F.col("stock_quantity") == 0, "FLAGGED").otherwise("VALID")
)

df_sales_silver = df_sales_b.withColumn(
    "sale_amount", F.when(F.col("sale_amount").isNull(), 0.0).otherwise(F.col("sale_amount").cast("double"))
).withColumn(
    "payment_mode", F.when((F.col("payment_mode").isNull()) | (F.trim(F.col("payment_mode")) == ""), "Not Provided").otherwise(F.col("payment_mode"))
).withColumn(
    "data_quality_status", F.when((F.col("sale_amount") == 0.0) | (F.col("payment_mode") == "Not Provided"), "FLAGGED").otherwise("VALID")
)

df_stores_silver = df_stores_b.withColumn("data_quality_status", F.lit("VALID"))

In [87]:
df_stores_silver.write.mode("overwrite").parquet("silver/stores")
df_products_silver.write.mode("overwrite").parquet("silver/products")
df_inventory_silver.write.mode("overwrite").parquet("silver/inventory")
df_sales_silver.write.mode("overwrite").parquet("silver/sales")

In [88]:
print("\n=== Part 3: JSON Flattening ===")


=== Part 3: JSON Flattening ===


In [89]:
df_suppliers_b = spark.read.parquet("bronze/suppliers")

In [90]:
df_suppliers_flat = df_suppliers_b.select(
    F.col("supplier_id"),
    F.col("supplier_name"),
    F.col("city"),
    F.col("rating"),
    F.col("contact.phone").alias("extracted_phone"),
    F.col("contact.email").alias("extracted_email")
)

In [91]:
df_suppliers_silver = df_suppliers_flat.withColumn(
    "phone", F.when(F.col("extracted_phone").isNull(), "Not Provided").otherwise(F.col("extracted_phone"))
).withColumn(
    "email", F.when(F.col("extracted_email").isNull(), "Not Provided").otherwise(F.col("extracted_email"))
).drop("extracted_phone", "extracted_email")

# 24. Save flattened suppliers data as Parquet
df_suppliers_silver.write.mode("overwrite").parquet("silver/suppliers")
print("Flattened Suppliers Schema:")
df_suppliers_silver.printSchema()

Flattened Suppliers Schema:
root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)



In [92]:
print("\n=== Part 4: Joins & Integrity Checking ===")


=== Part 4: Joins & Integrity Checking ===


In [93]:
stores = spark.read.parquet("silver/stores")
products = spark.read.parquet("silver/products")
inventory = spark.read.parquet("silver/inventory")
sales = spark.read.parquet("silver/sales")
suppliers = spark.read.parquet("silver/suppliers")

In [94]:
df_prod_supp = products.join(suppliers, on="supplier_id", how="left")
df_inv_prod = inventory.join(products, on="product_id", how="left")
df_sales_stores = sales.join(stores, on="store_id", how="left")
df_sales_prod = sales.join(products, on="product_id", how="left")

In [95]:
df_retail_sales = sales.join(stores, "store_id", "left") \
                       .join(products, "product_id", "left") \
                       .join(suppliers, "supplier_id", "left")

In [96]:
print("\n--- 30. Products with Invalid Supplier IDs ---")
products.join(suppliers, "supplier_id", "left_anti").show()


--- 30. Products with Invalid Supplier IDs ---
+-----------+----------+------------+-----------+---------+----------+-------------------+
|supplier_id|product_id|product_name|   category|    brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+---------+----------+-------------------+
|       S206|      P107|       Watch|    Fashion| Fastrack|      8000|              VALID|
|       S206|      P108|    Backpack|    Fashion|Wildcraft|      2500|              VALID|
|       S999|      P111|  Headphones|Electronics|     Sony|      3000|              VALID|
|    UNKNOWN|      P112|     T-Shirt|    Fashion|     Puma|      1500|            FLAGGED|
+-----------+----------+------------+-----------+---------+----------+-------------------+



In [97]:
print("\n--- 31. Inventory Rows with Invalid Product IDs ---")
inventory.join(products, "product_id", "left_anti").show()


--- 31. Inventory Rows with Invalid Product IDs ---
+----------+------------+--------+--------------+-------------+-----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|              VALID|
+----------+------------+--------+--------------+-------------+-----------+-------------------+



In [98]:
print("\n--- 33. Sales Rows with Invalid Store IDs ---")
sales.join(stores, "store_id", "left_anti").show()


--- 33. Sales Rows with Invalid Store IDs ---
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+



In [99]:
print("\n=== Part 5: Transformations ===")


=== Part 5: Transformations ===


In [100]:
df_inv_transformed = df_inv_prod.withColumn(
    "stock_status",
    F.when(F.col("stock_quantity") <= F.col("reorder_level"), "Reorder Required").otherwise("Sufficient Stock")
)

In [101]:
df_retail_transformed = df_retail_sales.withColumn(
    "price_category",
    F.when(F.col("unit_price") >= 50000, "Premium")
     .when(F.col("unit_price") >= 10000, "Standard")
     .otherwise("Budget")
).withColumn(
    "revenue_category",
    F.when(F.col("sale_amount") >= 50000, "High Revenue")
     .when(F.col("sale_amount") >= 15000, "Medium Revenue")
     .otherwise("Low Revenue")
)

In [102]:
df_retail_transformed = df_retail_transformed.withColumn(
    "month", F.month(F.col("sale_date"))
).withColumn(
    "year", F.year(F.col("sale_date"))
)

# 39. Valuation: Total inventory capital tied up in stock
df_inv_transformed = df_inv_transformed.withColumn(
    "inventory_value", F.col("stock_quantity") * F.col("unit_price")
)

In [103]:
df_suppliers_transformed = suppliers.withColumn(
    "supplier_quality",
    F.when(F.col("rating") >= 4.5, "Excellent")
     .when(F.col("rating") >= 4.0, "Good")
     .otherwise("Average")
)

In [104]:
print("\n=== Part 6: Aggregations ===")


=== Part 6: Aggregations ===


In [105]:
stores.groupBy("state").count().alias("stores_count").show(2)

+---------+-----+
|    state|count|
+---------+-----+
|Karnataka|    1|
|   Kerala|    1|
+---------+-----+
only showing top 2 rows


In [106]:
products.groupBy("category").count().show(2)
products.groupBy("brand").count().show(2)

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
+-----------+-----+
only showing top 2 rows
+-----+-----+
|brand|count|
+-----+-----+
| Nike|    1|
| Sony|    1|
+-----+-----+
only showing top 2 rows


In [107]:
df_inv_transformed.groupBy("store_id").agg(F.sum("inventory_value").alias("total_inv_value")).show(2)
df_inv_transformed.groupBy("category").agg(F.sum("inventory_value").alias("total_inv_value")).show(2)

+--------+---------------+
|store_id|total_inv_value|
+--------+---------------+
|    S105|         125000|
|    S102|         745000|
+--------+---------------+
only showing top 2 rows
+--------+---------------+
|category|total_inv_value|
+--------+---------------+
| Fashion|         292000|
|    NULL|           NULL|
+--------+---------------+
only showing top 2 rows


In [108]:
reorder_count = df_inv_transformed.filter(F.col("stock_status") == "Reorder Required").count()
print(f"Products Needing Reorder: {reorder_count}")

Products Needing Reorder: 5


In [110]:
print(f"Total Revenue Generated: {df_retail_transformed.select(F.sum('sale_amount')).collect()[0][0]}")
df_retail_transformed.groupBy("store_name").agg(F.sum("sale_amount").alias("revenue")).show(2)

df_retail_transformed.groupBy("category").agg(F.sum("sale_amount").alias("revenue")).show(2)
df_retail_transformed.groupBy("product_name").agg(F.sum("sale_amount").alias("revenue")).show(2)
df_retail_transformed.groupBy("payment_mode").agg(F.sum("sale_amount").alias("revenue")).show(2)

Total Revenue Generated: 373000.0
+--------------------+-------+
|          store_name|revenue|
+--------------------+-------+
|Metro Mart Bangalore|65000.0|
|    Metro Mart Kochi|32000.0|
+--------------------+-------+
only showing top 2 rows
+--------+-------+
|category|revenue|
+--------+-------+
| Fashion|62000.0|
|    NULL|10000.0|
+--------+-------+
only showing top 2 rows
+------------+-------+
|product_name|revenue|
+------------+-------+
|Office Chair|14000.0|
|        NULL|10000.0|
+------------+-------+
only showing top 2 rows
+------------+--------+
|payment_mode| revenue|
+------------+--------+
|        Card|133000.0|
|        Cash| 35500.0|
+------------+--------+
only showing top 2 rows


In [111]:
highest_product = df_retail_transformed.groupBy("product_name").agg(F.sum("sale_amount").alias("rev")).orderBy(F.desc("rev")).first()[0]
highest_store = df_retail_transformed.groupBy("store_name").agg(F.sum("sale_amount").alias("rev")).orderBy(F.desc("rev")).first()[0]
highest_category = df_retail_transformed.groupBy("category").agg(F.sum("sale_amount").alias("rev")).orderBy(F.desc("rev")).first()[0]
print(f"Top Product: {highest_product} | Top Store: {highest_store} | Top Category: {highest_category}")

Top Product: Laptop | Top Store: Metro Mart Hyderabad | Top Category: Electronics


In [112]:
print("\n=== Part 7: Window Functions ===")


=== Part 7: Window Functions ===


In [113]:
w_global_revenue = Window.orderBy(F.desc("total_revenue"))
w_category_revenue = Window.partitionBy("category").orderBy(F.desc("total_revenue"))
w_state_revenue = Window.partitionBy("state").orderBy(F.desc("total_revenue"))
w_running_revenue = Window.orderBy("sale_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_historical_sales = Window.partitionBy("product_id").orderBy("sale_date")

In [114]:
prod_rev_base = df_retail_transformed.groupBy("product_id", "product_name", "category").agg(F.sum("sale_amount").alias("total_revenue"))
store_rev_base = df_retail_transformed.groupBy("store_id", "store_name", "state").agg(F.sum("sale_amount").alias("total_revenue"))

In [115]:
df_prod_ranked = prod_rev_base.withColumn("rank", F.dense_rank().over(w_global_revenue))
df_store_ranked = store_rev_base.withColumn("rank", F.dense_rank().over(w_global_revenue))

In [116]:
# 58-60. Category-specific performance breakdown
df_cat_prod_ranked = prod_rev_base.withColumn("rank_in_cat", F.dense_rank().over(w_category_revenue))
df_top_prod_per_cat = df_cat_prod_ranked.filter(F.col("rank_in_cat") == 1)
df_top3_prod_per_cat = df_cat_prod_ranked.filter(F.col("rank_in_cat") <= 3)

In [117]:
df_top_store_state = df_store_ranked.withColumn("rank_in_state", F.dense_rank().over(w_state_revenue)).filter(F.col("rank_in_state") == 1)

In [118]:
daily_revenue = df_retail_transformed.groupBy("sale_date").agg(F.sum("sale_amount").alias("daily_rev"))
df_running_total = daily_revenue.withColumn("running_total", F.sum("daily_rev").over(w_running_revenue))

In [119]:
df_lag_lead = df_retail_transformed.select("product_id", "sale_date", "sale_amount") \
    .withColumn("prev_day_sale", F.lag("sale_amount", 1).over(w_historical_sales)) \
    .withColumn("next_day_sale", F.lead("sale_amount", 1).over(w_historical_sales)) \
    .withColumn("revenue_momentum", F.when(F.col("sale_amount") > F.col("prev_day_sale"), "INCREASED").otherwise("STABLE/DECREASED"))

print("Products with positive sales momentum compared to their last sale transaction:")
df_lag_lead.filter(F.col("revenue_momentum") == "INCREASED").show(3)

Products with positive sales momentum compared to their last sale transaction:
+----------+----------+-----------+-------------+-------------+----------------+
|product_id| sale_date|sale_amount|prev_day_sale|next_day_sale|revenue_momentum|
+----------+----------+-----------+-------------+-------------+----------------+
|      P107|2026-02-01|    16000.0|       8000.0|         NULL|       INCREASED|
+----------+----------+-----------+-------------+-------------+----------------+



In [120]:
print("\n=== Part 8: Spark SQL ===")


=== Part 8: Spark SQL ===


In [121]:
stores.createOrReplaceTempView("v_stores")
products.createOrReplaceTempView("v_products")
inventory.createOrReplaceTempView("v_inventory")
sales.createOrReplaceTempView("v_sales")
suppliers.createOrReplaceTempView("v_suppliers")

In [122]:
spark.sql("SELECT * FROM v_sales").show(2)
spark.sql("SELECT category, COUNT(*) as cnt FROM v_products GROUP BY category").show(2)
spark.sql("SELECT store_id, SUM(sale_amount) as rev FROM v_sales GROUP BY store_id").show(2)

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
| SA1001|    S101|      P101|2026-01-10|            1|    65000.0|         UPI|              VALID|
| SA1002|    S101|      P102|2026-01-10|            2|    50000.0|        Card|              VALID|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
only showing top 2 rows
+-----------+---+
|   category|cnt|
+-----------+---+
|    Fashion|  4|
|Electronics|  5|
+-----------+---+
only showing top 2 rows
+--------+-------+
|store_id|    rev|
+--------+-------+
|    S105|20000.0|
|    S102|65000.0|
+--------+-------+
only showing top 2 rows


In [123]:
spark.sql("SELECT * FROM v_inventory WHERE stock_quantity <= reorder_level").show(2)

+------------+--------+----------+--------------+-------------+-----------+-------------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+
|       I1003|    S101|      P104|             3|            5| 2026-01-11|              VALID|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|              VALID|
+------------+--------+----------+--------------+-------------+-----------+-------------------+
only showing top 2 rows


In [126]:
spark.sql("SELECT * FROM v_sales WHERE product_id NOT IN (SELECT product_id FROM v_products)").show()
spark.sql("SELECT * FROM v_products WHERE supplier_id NOT IN (SELECT supplier_id FROM v_suppliers) AND supplier_id != 'UNKNOWN'").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+
| SA1009|    S108|      P120|2026-01-15|            2|    10000.0|        Cash|              VALID|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+

+----------+------------+-----------+---------+-----------+----------+-------------------+
|product_id|product_name|   category|    brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+---------+-----------+----------+-------------------+
|      P107|       Watch|    Fashion| Fastrack|       S206|      8000|              VALID|
|      P108|    Backpack|    Fashion|Wildcraft|       S206|      2500|              VALID|
|      P111|  Headphones|Electronics|     So

In [127]:
spark.sql("SELECT product_id, SUM(sale_amount) as rev FROM v_sales GROUP BY product_id ORDER BY rev DESC LIMIT 5").show()
spark.sql("SELECT payment_mode, SUM(sale_amount) as rev FROM v_sales GROUP BY payment_mode").show()

+----------+--------+
|product_id|     rev|
+----------+--------+
|      P101|130000.0|
|      P102| 75000.0|
|      P109| 38000.0|
|      P110| 32000.0|
|      P107| 24000.0|
+----------+--------+

+------------+--------+
|payment_mode|     rev|
+------------+--------+
|        Card|133000.0|
|        Cash| 35500.0|
|Not Provided| 14000.0|
|         UPI|190500.0|
+------------+--------+



In [128]:
print("\n=== Part 9: Full Refresh and Incremental Load ===")


=== Part 9: Full Refresh and Incremental Load ===


In [130]:
# Re-define the unified core dataframe with renamed city columns
df_retail_sales = sales \
    .join(stores.withColumnRenamed("city", "store_city"), "store_id", "left") \
    .join(products, "product_id", "left") \
    .join(suppliers.withColumnRenamed("city", "supplier_city"), "supplier_id", "left")

In [125]:
with open("raw_data/sales_march_incremental.csv", "w") as f:
    f.write("""sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_mode
SA1016,S101,P101,2026-03-01,2,130000,UPI
SA1017,S102,P102,2026-03-02,1,25000,Card""")

In [131]:
df_incr_raw = spark.read.csv("raw_data/sales_march_incremental.csv", header=True, inferSchema=True)
df_incr_silver = df_incr_raw.withColumn(
    "sale_amount", F.when(F.col("sale_amount").isNull(), 0.0).otherwise(F.col("sale_amount").cast("double"))
).withColumn(
    "payment_mode", F.when((F.col("payment_mode").isNull()) | (F.trim(F.col("payment_mode")) == ""), "Not Provided").otherwise(F.col("payment_mode"))
).withColumn("data_quality_status", F.lit("VALID"))

In [132]:
df_incr_silver.write.mode("append").parquet("silver/sales")

In [133]:
sales_updated = spark.read.parquet("silver/sales")
df_retail_updated = sales_updated.join(stores, "store_id", "left") \
                                 .join(products, "product_id", "left") \
                                 .join(suppliers, "supplier_id", "left") \
                                 .withColumn("year", F.year(F.col("sale_date"))) \
                                 .withColumn("month", F.month(F.col("sale_date")))

In [135]:
print(f"Sales count prior to batch: {sales.count()}")
print(f"Sales count post batch append: {sales_updated.count()}")

Sales count prior to batch: 15
Sales count post batch append: 17


In [144]:
my_drive_path = "/content/drive/MyDrive"

if os.path.exists(my_drive_path):
    print("Items found in your Google Drive:")
    print(os.listdir(my_drive_path))
else:
    print("Can't access MyDrive. Is it named something else in your region?")

Items found in your Google Drive:
['IMG_20221103_160023.jpg', 'IMG_20221104_200114~2 (1).jpg', 'IMG_20221104_200114~2.jpg', 'IMG_20221104_200335~2.jpg', 'Classroom', 'IMG-20230610-WA0005.jpg', 'flood prediction.gsheet', 'DFO_2507_From_20040620_to_20041007.zip', 'Dark Blue And Pink Creative Gradient Business Project Presentation (1).ppt.gslides', 'hackthon.gslides', 'Dark Blue And Pink Creative Gradient Business Project Presentation.ppt.gslides', 'ep project.gdoc', 'Screenshot_2023_1002_105509.jpg', 'Project proposal.gdoc', 'CS2303_LAB_PROJECTS_UPDATED.gdoc', 'CS2303_FRONT_INDEX.gdoc', 'IMG-20231021-WA0032.jpg', 'Screenshot_2023_1026_103340.jpg', 'Offer Letter.pdf', 'Colab Notebooks', 'sword_in_the_stone_v001.blend', 'money.jpg', 'Untitled document.gdoc', 'Icpro.pdf', 'UNIT I Introduction (1).gslides', 'UNIT I Introduction.gslides', 'UNIT I  Search.gslides', 'swaraj.pdf', 'wepik-revolutionizing-e-bikes-exploring-the-potential-of-super-capacitors-as-power-sources-20240131141416ZHgY.pptx'